# Tier 2: Is there a better dataset than FakeHealth?

A TF-IDF reality check, a dataset quality probe, ReCOVery as a candidate corpus, and a test of using more articles. Every number is read from saved result files in `outputs/`.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 130)
ROOT = Path.cwd(); OUTP = ROOT / "outputs"

leak   = json.load(open(OUTP / "dataset_leakage_probe.json"))
tfidf  = json.load(open(OUTP / "tier2_tfidf_canonical.json"))
recmet = json.load(open(OUTP / "recovery_meta.json"))
trans  = pd.read_csv(OUTP / "tier2_transfer_results.csv")
cv     = pd.read_csv(OUTP / "tier2_cv_results.csv")
print("artefacts loaded")

artefacts loaded


## 1. Does the transformer beat TF-IDF?

TF-IDF with logistic regression on the same split, with the same rate matched threshold.

In [2]:
ref = pd.DataFrame([
    {"system": "TF-IDF + logistic regression", **{k: tfidf[k] for k in
        ("f1_macro","accuracy","recall_notcred","roc_auc")}},
    {"system": "v1 binary PubMedBERT (8-seed)", "f1_macro": 0.6582, "accuracy": 0.7036,
     "recall_notcred": 0.5058, "roc_auc": 0.7009},
    {"system": "v2 ordinal+criteria (8-seed)", "f1_macro": 0.6851, "accuracy": 0.7229,
     "recall_notcred": 0.5596, "roc_auc": 0.7179},
])
print(ref.to_string(index=False))

                       system  f1_macro  accuracy  recall_notcred  roc_auc
 TF-IDF + logistic regression    0.6663    0.7107          0.5140   0.7268
v1 binary PubMedBERT (8-seed)    0.6582    0.7036          0.5058   0.7009
 v2 ordinal+criteria (8-seed)    0.6851    0.7229          0.5596   0.7179


## 2. Dataset quality probe

Do outlets carry both labels, and how far does TF-IDF fall when train and test share no outlet?

In [3]:
fh, rc = leak["fakehealth"], leak["recovery"]
tbl = pd.DataFrame([
    {"": "outlets", "FakeHealth": fh["n_sources_total"], "ReCOVery": rc["n_publishers"]},
    {"": "outlets carrying BOTH labels",
     "FakeHealth": f"{fh['sources_with_both_labels']} of {fh['n_sources_total']}",
     "ReCOVery": f"{rc['publishers_with_both_labels']} of {rc['n_publishers']}"},
    {"": "label is a function of the outlet", "FakeHealth": "no", "ReCOVery": "YES"},
    {"": "TF-IDF, random split", "FakeHealth": fh["tfidf_random"], "ReCOVery": rc["tfidf_random"]},
    {"": "TF-IDF, outlet-disjoint split",
     "FakeHealth": fh["tfidf_source_disjoint"], "ReCOVery": rc["tfidf_publisher_disjoint"]},
    {"": "inflation from outlet memorisation",
     "FakeHealth": f"+{fh['drop']:.3f}", "ReCOVery": f"+{rc['drop']:.3f}"},
])
print(tbl.to_string(index=False))

                                   FakeHealth ReCOVery
                           outlets        450       55
      outlets carrying BOTH labels  83 of 450  0 of 55
 label is a function of the outlet         no      YES
              TF-IDF, random split   0.665929    0.904
     TF-IDF, outlet-disjoint split   0.611387    0.773
inflation from outlet memorisation     +0.055   +0.131


## 3. Training on ReCOVery

Publisher disjoint splits.

In [4]:
r1 = trans[trans.exp == "R1_recovery"]
per, ens = r1[r1.seed != "ENS"], r1[r1.seed == "ENS"]
print(f"ReCOVery: {recmet['n_articles']} articles, {recmet['n_publishers']} publishers, "
      f"majority baseline {recmet['majority_baseline']:.3f}\n")
print(pd.DataFrame([
    {"system": "TF-IDF (publisher-disjoint)", "macro_f1": 0.773, "accuracy": None, "roc_auc": None},
    {"system": "PubMedBERT (3-seed mean)", "macro_f1": round(per.f1_macro.mean(),4),
     "accuracy": round(per.accuracy.mean(),4), "roc_auc": round(per.roc_auc.mean(),4)},
    {"system": "PubMedBERT (seed ensemble)", "macro_f1": ens.f1_macro.iloc[0],
     "accuracy": ens.accuracy.iloc[0], "roc_auc": ens.roc_auc.iloc[0]},
]).to_string(index=False))

ReCOVery: 2013 articles, 55 publishers, majority baseline 0.675

                     system  macro_f1  accuracy  roc_auc
TF-IDF (publisher-disjoint)    0.7730       NaN      NaN
   PubMedBERT (3-seed mean)    0.7237    0.7508   0.8333
 PubMedBERT (seed ensemble)    0.7313    0.7574   0.8408


### Transfer test: a FakeHealth model scored on ReCOVery

In [5]:
z = trans[trans.exp == "R2_fh_to_rec"]
print(z[["score","f1_macro","accuracy","roc_auc"]].to_string(index=False))
print("\nROC-AUC 0.50 = chance.")

score  f1_macro  accuracy  roc_auc
  ord    0.4939    0.5279   0.5039
 crit    0.4600    0.5049   0.4864

ROC-AUC 0.50 = chance.


### Can ReCOVery donate data?

The encoder is trained on ReCOVery first, then fine tuned with the v2 heads on FakeHealth.

In [6]:
r3 = trans[trans.exp == "R3_intermediate"]
print(r3.groupby("score")[["f1_macro","accuracy","recall_notcred","roc_auc"]]
        .agg(["mean","std"]).round(4).to_string())
print("\ndirect v2 reference (8 seeds): crit_lr macro-F1 0.6851 +- 0.0160")

          f1_macro         accuracy         recall_notcred         roc_auc        
              mean     std     mean     std           mean     std    mean     std
score                                                                             
crit        0.6819  0.0174   0.7191  0.0214         0.5607  0.0247  0.7132  0.0051
crit_lr     0.6762  0.0111   0.7149  0.0101         0.5483  0.0143  0.7123  0.0025
crit_nodm   0.6786  0.0150   0.7159  0.0149         0.5576  0.0235  0.7101  0.0049
ord         0.6779  0.0202   0.7170  0.0196         0.5483  0.0300  0.7155  0.0003

direct v2 reference (8 seeds): crit_lr macro-F1 0.6851 +- 0.0160


## 4. Does using more articles help?

The validation split is pooled with training (1,483 to 1,801 articles) under 5 fold cross validation. The test set is untouched.

In [7]:
print(cv[["score","accuracy","f1_macro","recall_notcred","roc_auc"]].to_string(index=False))
print("\nreference, v2 trained on train only (8-seed mean, same rule):")
print("  crit_lr    acc=0.7229  macroF1=0.6851  recall=0.5596  auc=0.7179")

    score  accuracy  f1_macro  recall_notcred  roc_auc
      ord    0.7138    0.6803          0.5794   0.7091
     crit    0.7044    0.6748          0.5981   0.7093
crit_nodm    0.6981    0.6705          0.6075   0.7082
  crit_lr    0.7170    0.6859          0.5981   0.7133

reference, v2 trained on train only (8-seed mean, same rule):
  crit_lr    acc=0.7229  macroF1=0.6851  recall=0.5596  auc=0.7179
